Step 1 — Load and inspect the dataset
Open your Python environment or Jupyter Notebook and make sure Dataset 2 is available. Run the following code.

In [4]:
import pandas as pd

# Load the CareFlow dataset
df = pd.read_csv("Careflowdataset.csv")

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

Dataset shape: (9855, 9)

Columns:
['Case ID', 'Event Index', 'Activity', 'Timestamp', 'Next Activity', 'Elapsed to Next (min)', 'Resource', 'Department', 'Status']

First 5 rows:
         Case ID  Event Index                      Activity  \
0  PAT-2026-9840            1      ED Registration & Triage   
1  PAT-2026-9840            2  Initial Physician Assessment   
2  PAT-2026-9840            3  Stat Blood Draw & Biomarkers   
3  PAT-2026-9644            1      ED Registration & Triage   
4  PAT-2026-9644            2  Initial Physician Assessment   

                  Timestamp                    Next Activity  \
0  2026-09-12T08:00:00.000Z     Initial Physician Assessment   
1  2026-09-12T08:10:00.000Z     Stat Blood Draw & Biomarkers   
2  2026-09-12T08:25:00.000Z  Central Laboratory Specimen Run   
3  2026-09-12T08:28:00.000Z     Initial Physician Assessment   
4  2026-09-12T08:52:00.000Z     Stat Blood Draw & Biomarkers   

   Elapsed to Next (min)                 Resource       

Step 2 — Verify event ordering
We need to check whether each patient's events are ordered correctly by Event Index, and whether their timestamps are chronological.

How to interpret the results:
- Duplicate event indices should be investigated.
- Out-of-order timestamps should be investigated.
- A zero result for both checks is a good sign, but it doesn't by itself prove that the entire event log is correct.

In [5]:
# Sort events within each patient case
df = df.sort_values(
    by=["Case ID", "Event Index"]
).reset_index(drop=True)

# Check for duplicate event indices within a case
duplicate_events = df.duplicated(
    subset=["Case ID", "Event Index"],
    keep=False
)

print("Rows with duplicate event indices:",
      duplicate_events.sum())

# Check whether timestamps are chronological within each case
df["Timestamp"] = pd.to_datetime(df["Timestamp"], utc=True)

timestamp_diff = (
    df.groupby("Case ID")["Timestamp"].diff()
)

out_of_order = timestamp_diff < pd.Timedelta(0)

print("Cases with out-of-order timestamps:",
      df.loc[out_of_order, "Case ID"].nunique())

print("Events with out-of-order timestamps:",
      out_of_order.sum())

Rows with duplicate event indices: 0
Cases with out-of-order timestamps: 0
Events with out-of-order timestamps: 0


Step 3 — Verify the activity sequence
Now we will compare the Next Activity column with the activity actually recorded in the next event.

In [6]:
# Get the next recorded activity within each case
df["Actual Next Activity"] = (
    df.groupby("Case ID")["Activity"].shift(-1)
)

# Compare the recorded next activity with the actual next event
has_next = df["Actual Next Activity"].notna()

matches = (
    df.loc[has_next, "Next Activity"]
    == df.loc[has_next, "Actual Next Activity"]
)

print("Events with a subsequent event:", has_next.sum())
print("Matching next activities:", matches.sum())
print("Mismatches:", (~matches).sum())

# Display mismatches for investigation
mismatches = df.loc[
    has_next & (df["Next Activity"] != df["Actual Next Activity"]),
    ["Case ID", "Event Index", "Activity",
     "Next Activity", "Actual Next Activity"]
]

print(mismatches.head(20))

Events with a subsequent event: 8427
Matching next activities: 8427
Mismatches: 0
Empty DataFrame
Columns: [Case ID, Event Index, Activity, Next Activity, Actual Next Activity]
Index: []


Step 4 — Display complete patient journeys
Let's create a readable activity sequence for each case.

In [7]:
journeys = (
    df.groupby("Case ID")["Activity"]
    .agg(list)
)

print("Sample patient journeys:\n")

for case_id, activities in journeys.head(10).items():
    print(f"{case_id}: {' -> '.join(activities)}")

Sample patient journeys:

PAT-2026-10000: ED Registration & Triage -> Initial Physician Assessment -> Stat Blood Draw & Biomarkers -> Central Laboratory Specimen Run -> Empiric Antibiotic / Targeted Therapy -> Inpatient Ward Bed Handover
PAT-2026-10001: ED Registration & Triage -> Initial Physician Assessment -> Stat Blood Draw & Biomarkers -> Central Laboratory Specimen Run -> Empiric Antibiotic / Targeted Therapy -> Inpatient Ward Bed Handover
PAT-2026-10002: ED Registration & Triage -> Initial Physician Assessment -> Stat Blood Draw & Biomarkers -> Central Laboratory Specimen Run -> Diagnostic CT / Angiography Imaging -> Sub-Specialist On-Call Consult -> Empiric Antibiotic / Targeted Therapy -> Direct ICU Bed Transfer
PAT-2026-10003: ED Registration & Triage -> Initial Physician Assessment -> Stat Blood Draw & Biomarkers -> Central Laboratory Specimen Run -> Empiric Antibiotic / Targeted Therapy -> Inpatient Ward Bed Handover
PAT-2026-10004: ED Registration & Triage -> Initial Physi